# Workflow 13 - Article figures (uniform, publication-quality)
## BRISC 2025 (audited) - all approaches from one dataset

Generates every figure for the article with one shared style: consistent fonts, a fixed colour palette per approach family, legends placed to the right of each panel, 300 dpi, and bootstrap 95% CIs as error bars. Statistical tests are added where valid: **McNemar** for paired classifiers on the *same* 678-image test set (predictions regenerated in a fixed order so they align by image), and **Friedman + Nemenyi** across approaches on per-class F1.

All numbers trace to `leakage_backbones/`, `w11_hybrid/`, `classic_ml_all_results.csv`, `w11_hybrid/shap/`. Figures are written to `figuras_artigo/`.

## 0. Shared style + paths

In [ ]:
import os, io, json, pickle, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

PROJ = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui'
FIG  = f'{PROJ}/figuras_artigo'; os.makedirs(FIG, exist_ok=True)
AUD  = '/root/brisc/data/brisc2025_clean/classification_task'
CLASSES = ['glioma','meningioma','no_tumor','pituitary']

# --- one consistent style for every figure ---
mpl.rcParams.update({
    'figure.dpi':120, 'savefig.dpi':300, 'font.size':10, 'font.family':'DejaVu Sans',
    'axes.titlesize':11, 'axes.titleweight':'bold', 'axes.labelsize':10,
    'xtick.labelsize':9, 'ytick.labelsize':9, 'legend.fontsize':9,
    'axes.spines.top':False, 'axes.spines.right':False, 'axes.grid':True,
    'grid.alpha':0.25, 'grid.linewidth':0.6,
})
# colour per approach family (colour-blind friendly)
FAMILY_COLOR = {
    'Classical ML':'#8C6D31', 'CNN scratch':'#7A7A9D', 'Transfer (softmax)':'#1F77B4',
    'Medical pretrain':'#D62728', 'Hybrid':'#1F4E4A',
}

def save(fig, name):
    fig.savefig(f'{FIG}/{name}.png', bbox_inches='tight')
    fig.savefig(f'{FIG}/{name}.pdf', bbox_inches='tight')
    fig.savefig(f'{FIG}/{name}.svg', bbox_inches='tight')   # editable vector
    print('saved', name)
print('style set; output ->', FIG)

## 1. Figure 1 - Workflow schematic (all approaches from one dataset)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.2)); ax.set_xlim(0,12); ax.set_ylim(0,7); ax.axis('off')
def box(x,y,w,h,text,color,tc='white',fs=9):
    ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle='round,pad=0.06,rounding_size=0.12',
        fc=color,ec='black',lw=1.1))
    ax.text(x+w/2,y+h/2,text,ha='center',va='center',color=tc,fontsize=fs,fontweight='bold')
def arrow(x1,y1,x2,y2):
    ax.add_patch(FancyArrowPatch((x1,y1),(x2,y2),arrowstyle='-|>',mutation_scale=13,lw=1.1,color='#333'))
# dataset
box(0.2,2.7,2.1,1.6,'BRISC 2025\naudited\n(MD5 + pHash)\n5041 imgs',' #333333'.strip(),fs=9)
# branches
box(3.0,5.3,3.0,1.1,'Manual features\nwavelet + GLCM (198)',FAMILY_COLOR['Classical ML'],fs=9)
box(3.0,3.9,3.0,1.0,'CNN from scratch\n(3 conv blocks)',FAMILY_COLOR['CNN scratch'],fs=9)
box(3.0,2.4,3.0,1.1,'Transfer backbones\nResNet50 / VGG16 /\nEffNetB2 / ConvNeXt',FAMILY_COLOR['Transfer (softmax)'],fs=8.5)
box(3.0,1.0,3.0,1.0,'RadImageNet\n(medical pretrain)',FAMILY_COLOR['Medical pretrain'],fs=9)
for yy in (5.85,4.4,2.95,1.5): arrow(2.3,3.5,3.0,yy)
# heads
box(6.6,5.3,2.3,1.1,'19 classifiers\n(LGBM/XGB/RF/...)',FAMILY_COLOR['Classical ML'],fs=8.5)
box(6.6,3.9,2.3,1.0,'softmax head',FAMILY_COLOR['CNN scratch'],fs=9)
box(6.6,2.4,2.3,1.1,'softmax  +\nhybrid (7 clf)',FAMILY_COLOR['Hybrid'],fs=8.5)
box(6.6,1.0,2.3,1.0,'softmax + hybrid',FAMILY_COLOR['Hybrid'],fs=8.5)
for y in (5.85,4.4,2.95,1.5): arrow(6.0,y,6.6,y)
# eval
box(9.4,2.6,2.4,1.8,'Same 678-img\ntest set\n\nAcc / F1 / AUC\nbootstrap CI\nMcNemar\nGrad-CAM + SHAP','#333333',fs=8.5)
for y in (5.85,4.4,2.95,1.5): arrow(8.9,y,9.4,3.5)
ax.set_title('All approaches evaluated on the same audited BRISC 2025 partition', fontsize=12)
save(fig,'fig1_workflow_schematic'); plt.show()

## 2. Figure 2 - Grand comparison of approach families (bootstrap 95% CI)

In [ ]:
import csv
def load_csv(p): return list(csv.DictReader(io.open(p,encoding='utf-8')))
lk = {(r['workflow'],r['dataset']):r for r in load_csv(f'{PROJ}/leakage_backbones/all_results.csv')}
hy = load_csv(f'{PROJ}/w11_hybrid/all_hybrid_results.csv')
cl = [r for r in load_csv(f'{PROJ}/classic_ml_all_results.csv') if r['split']=='original']
def fnum(x):
    try: return float(x)
    except: return np.nan
best_cl = max(cl, key=lambda r: fnum(r['accuracy']))
def hyb(bb,clf):
    for r in hy:
        if r['backbone']==bb and r['classifier']==clf: return r
# rows: (label, family, acc, ci_lo, ci_hi)
w7h = hyb('W7_VGG16_ImageNet','LightGBM')
rows = [
  ('Classical ML\n(db4+GLCM+LGBM)','Classical ML',90.71,None,None),
  ('CNN scratch\n(W2)','CNN scratch',93.81,91.15,94.69),
  ('Transfer softmax\n(VGG16)','Transfer (softmax)',97.94,96.61,98.97),
  ('Medical pretrain\n(RadImageNet)','Medical pretrain',72.12,69.17,75.37),
  ('Hybrid\n(VGG16+LightGBM)','Hybrid',float(w7h['accuracy']),float(w7h['ci95_lo']),float(w7h['ci95_hi'])),
]
fig, ax = plt.subplots(figsize=(9,4.6))
xs = np.arange(len(rows))
for i,(lab,fam,acc,lo,hi) in enumerate(rows):
    yerr = None if lo is None else [[acc-lo],[hi-acc]]
    ax.bar(i, acc, 0.62, color=FAMILY_COLOR[fam], edgecolor='black', lw=0.7,
           yerr=yerr, capsize=4, ecolor='#222')
    ax.text(i, acc+ (0 if lo is None else (hi-acc)) +0.6, f'{acc:.1f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(xs); ax.set_xticklabels([r[0] for r in rows], fontsize=8.5)
ax.set_ylabel('Test accuracy (%)'); ax.set_ylim(65,102)
ax.set_title('Approach families on the same audited BRISC 2025 test set (n=678)')
# caption/legend beside
handles=[plt.Rectangle((0,0),1,1,color=c) for c in FAMILY_COLOR.values()]
ax.legend(handles, list(FAMILY_COLOR.keys()), title='Approach family',
          loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False)
save(fig,'fig2_grand_comparison'); plt.show()

## 3. Figure 3 - Softmax head vs best hybrid, per backbone

In [ ]:
soft = {'W3_CNN_scratch':93.81,'W6_ResNet50_ImageNet':97.49,'W7_VGG16_ImageNet':97.94,
        'W8_EfficientNetB2_ImageNet':93.95,'W9_ConvNeXtTiny_ImageNet':94.99,'W10_RadImageNet_ResNet50':72.12}
order = ['W7_VGG16_ImageNet','W6_ResNet50_ImageNet','W9_ConvNeXtTiny_ImageNet',
         'W8_EfficientNetB2_ImageNet','W3_CNN_scratch','W10_RadImageNet_ResNet50']
best_h = {}
for r in hy:
    b=r['backbone']
    if b not in best_h or fnum(r['accuracy'])>fnum(best_h[b]['accuracy']): best_h[b]=r
labels=[b.replace('_ImageNet','').replace('_',' ') for b in order]
s=[soft[b] for b in order]; h=[float(best_h[b]['accuracy']) for b in order]
fig, ax = plt.subplots(figsize=(9.5,4.6)); x=np.arange(len(order)); w=0.38
ax.bar(x-w/2, s, w, color='#B0B0C0', edgecolor='black', lw=0.6, label='softmax head')
ax.bar(x+w/2, h, w, color=FAMILY_COLOR['Hybrid'], edgecolor='black', lw=0.6, label='best hybrid')
for i,b in enumerate(order):
    d=h[i]-s[i]; ax.text(x[i], max(h[i],s[i])+0.8, f'{d:+.1f}', ha='center', fontsize=8.5, color='#B22')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=8.5)
ax.set_ylabel('Test accuracy (%)'); ax.set_ylim(65,102)
ax.set_title('Hybrid gain over softmax (audited); delta annotated in red')
ax.legend(loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False)
save(fig,'fig3_softmax_vs_hybrid'); plt.show()

## 4. Figure 4 - Leakage delta per backbone (audited vs unaudited)

In [ ]:
bbs=['W6_ResNet50_ImageNet','W7_VGG16_ImageNet','W8_EfficientNetB2_ImageNet','W9_ConvNeXtTiny_ImageNet','W10_RadImageNet_ResNet50']
fig, ax = plt.subplots(figsize=(9.5,4.6)); x=np.arange(len(bbs)); w=0.38
for k,ds in enumerate(['audited','unaudited']):
    accs=[float(lk[(b,ds)]['accuracy']) for b in bbs]
    los=[float(lk[(b,ds)]['accuracy'])-float(lk[(b,ds)]['ci95_lo']) for b in bbs]
    his=[float(lk[(b,ds)]['ci95_hi'])-float(lk[(b,ds)]['accuracy']) for b in bbs]
    ax.bar(x+(k-0.5)*w, accs, w, yerr=[los,his], capsize=3, ecolor='#222',
           color='#1F4E4A' if ds=='audited' else '#7FB3AE', edgecolor='black', lw=0.6, label=ds)
ax.set_xticks(x); ax.set_xticklabels([b.replace('_ImageNet','').replace('_',' ') for b in bbs], rotation=15, ha='right', fontsize=8.5)
ax.set_ylabel('Test accuracy (%)'); ax.set_ylim(65,102)
ax.set_title('Leakage check: audited vs unaudited (95% bootstrap CI). All CIs overlap.')
ax.legend(loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False, title='partition')
save(fig,'fig4_leakage_delta'); plt.show()

## 5. Figure 5 - Classical ML: factorial effects and official vs random split

In [ ]:
clA = load_csv(f'{PROJ}/classic_ml_all_results.csv')
def best_per(split):
    d={}
    for r in clA:
        if r['split']!=split: continue
        s=r['scenario']
        if s not in d or fnum(r['accuracy'])>fnum(d[s]['accuracy']): d[s]=r
    return d
bo=best_per('original'); br=best_per('aleatoria')
def is_ss(s): return 'no_ss' not in s
def is_fs(s): return s.endswith('_fs') and 'no_fs' not in s
def gmean(dct,pred): 
    v=[fnum(dct[s]['accuracy']) for s in dct if pred(s)]; return np.mean(v) if v else np.nan
fig, axs = plt.subplots(1,2, figsize=(11,4.4))
# panel A: factorial effects (original)
eff=[('no SS',gmean(bo,lambda s:not is_ss(s))),('SS',gmean(bo,is_ss)),
     ('no FS',gmean(bo,lambda s:not is_fs(s))),('FS',gmean(bo,is_fs))]
axs[0].bar([e[0] for e in eff],[e[1] for e in eff],color=['#4C9A2A','#C0392B','#4C9A2A','#C0392B'],edgecolor='black',lw=0.6)
axs[0].set_ylim(80,92); axs[0].set_ylabel('mean best accuracy (%)')
axs[0].set_title('Factorial effects (audited/original split)')
axs[0].text(0.5,81,f"SS: {eff[1][1]-eff[0][1]:+.2f} pp",ha='center',fontsize=9)
axs[0].text(2.5,81,f"FS: {eff[3][1]-eff[2][1]:+.2f} pp",ha='center',fontsize=9)
# panel B: official vs random, best scenario per split
scen=sorted(set(bo)&set(br), key=lambda s:-fnum(bo[s]['accuracy']))[:6]
x=np.arange(len(scen)); w=0.38
axs[1].bar(x-w/2,[fnum(bo[s]['accuracy']) for s in scen],w,color='#1F4E4A',edgecolor='black',lw=0.6,label='original (audited)')
axs[1].bar(x+w/2,[fnum(br[s]['accuracy']) for s in scen],w,color='#C99A2E',edgecolor='black',lw=0.6,label='random')
axs[1].set_xticks(x); axs[1].set_xticklabels([s.replace('_',' ') for s in scen], rotation=25, ha='right', fontsize=7.5)
axs[1].set_ylim(80,94); axs[1].set_ylabel('best accuracy (%)')
axs[1].set_title('Official vs random split (classical ML)')
axs[1].legend(loc='center left', bbox_to_anchor=(1.01,0.5), frameon=False)
fig.suptitle('Classical ML (hand-crafted wavelet+GLCM features)', y=1.02, fontsize=12, fontweight='bold')
save(fig,'fig5_classical_factorial_split'); plt.show()

## 6. Figure 6 - SHAP: within-backbone agreement and concentration

In [ ]:
wa = load_csv(f'{PROJ}/w11_hybrid/shap/within_backbone_agreement.csv')
conc = pd.read_csv(f'{PROJ}/w11_hybrid/shap/top10_concentration.csv', index_col=0)
fig, axs = plt.subplots(1,2, figsize=(11,4.4))
wa_s=sorted(wa, key=lambda r: float(r['mean_rho_across_classifiers']))
axs[0].barh([r['backbone'].replace('_ImageNet','').replace('_',' ') for r in wa_s],
            [float(r['mean_rho_across_classifiers']) for r in wa_s], color='#1F4E4A', edgecolor='black', lw=0.6)
axs[0].set_xlim(0,1); axs[0].axvline(0.5, ls='--', color='grey', lw=0.8)
axs[0].set_xlabel('mean pairwise Spearman rho'); axs[0].set_title('Within-backbone agreement (4 classifiers)')
im=axs[1].imshow(conc.values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
axs[1].set_xticks(range(len(conc.columns))); axs[1].set_xticklabels(conc.columns, rotation=25, ha='right', fontsize=8)
axs[1].set_yticks(range(len(conc.index))); axs[1].set_yticklabels([i.replace('_ImageNet','').replace('_',' ') for i in conc.index], fontsize=7.5)
for i in range(conc.shape[0]):
    for j in range(conc.shape[1]):
        axs[1].text(j,i,f'{conc.values[i,j]:.2f}',ha='center',va='center',fontsize=7,color='white' if conc.values[i,j]<0.6 else 'black')
fig.colorbar(im, ax=axs[1], label='top-10 / total |SHAP|', shrink=0.8)
axs[1].set_title('Decision concentration'); axs[1].grid(False)
save(fig,'fig6_shap'); plt.show()

## 7. Statistical tests - regenerate aligned predictions, then McNemar

Predictions are regenerated in a fixed image order (with filenames) so paired McNemar is valid. This loads the trained models (TF) plus the classical LGBM re-fit on the MATLAB db4/NL16 features.

In [ ]:
# Build the audited test dataframe in a FIXED order (sorted by path) -> shared index.
import glob
def test_paths():
    rows=[]
    for c in CLASSES:
        for fn in sorted(os.listdir(f'{AUD}/test/{c}')):
            if fn.lower().endswith(('.jpg','.jpeg','.png')): rows.append((f'{AUD}/test/{c}/{fn}', c, fn))
    return rows
TP = test_paths(); print('test images:', len(TP))
cls_idx={c:i for i,c in enumerate(CLASSES)}
y_true = np.array([cls_idx[c] for _,c,_ in TP])
pred_store = {}  # approach -> y_pred aligned to TP

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
for g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(g,True)
    except Exception: pass
try:
    from keras.src.applications.convnext import LayerScale; CO={'LayerScale':LayerScale}
except Exception: CO={}

def preprocess(name):
    if name=='rescale': return lambda x:x/255.0
    from tensorflow.keras.applications.vgg16 import preprocess_input as v; 
    if name=='vgg16': return v
    from tensorflow.keras.applications.resnet50 import preprocess_input as r
    if name in ('resnet50','radimagenet_ignore'): return r
    return lambda x:x/255.0

def predict_softmax(model_path, img_size, pre, custom=None):
    m=load_model(model_path, compile=False, custom_objects=custom or {})
    out=np.zeros(len(TP),dtype=int)
    B=32
    for i in range(0,len(TP),B):
        batch=TP[i:i+B]
        X=np.stack([pre(img_to_array(load_img(p,target_size=(img_size,img_size)))) for p,_,_ in batch]).astype('float32')
        out[i:i+len(batch)]=np.argmax(m.predict(X,verbose=0),axis=1)
    del m; tf.keras.backend.clear_session(); return out
print('inference helpers ready')

In [ ]:
# Headline models (softmax): CNN scratch (128,/255), VGG16 (224,vgg), RadImageNet (224,resnet)
pred_store['CNN scratch'] = predict_softmax(f'{PROJ}/modelo_brisc.h5',128,preprocess('rescale'))
from tensorflow.keras.applications.vgg16 import preprocess_input as pi_vgg
pred_store['Transfer softmax (VGG16)'] = predict_softmax(f'{PROJ}/leakage_backbones/W7_VGG16_ImageNet/audited/model.h5',224,pi_vgg)
pred_store['Medical pretrain (RadImageNet)'] = predict_softmax(f'{PROJ}/leakage_backbones/W10_RadImageNet_ResNet50/audited/model.h5',224,lambda x:(x/127.5)-1.0)
print({k:(v==y_true).mean()*100 for k,v in pred_store.items()})

In [ ]:
# Hybrid VGG16 + LightGBM: extract feat_dense then classify
from tensorflow.keras.applications.vgg16 import preprocess_input as pi_vgg
def deep_feats(model_path, img_size, pre, custom=None, layer='feat_dense'):
    m=load_model(model_path, compile=False, custom_objects=custom or {})
    ext=Model(m.input, m.get_layer(layer).output)
    F=np.zeros((len(TP),ext.output_shape[-1]),dtype='float32'); B=32
    for i in range(0,len(TP),B):
        batch=TP[i:i+B]
        X=np.stack([pre(img_to_array(load_img(p,target_size=(img_size,img_size)))) for p,_,_ in batch]).astype('float32')
        F[i:i+len(batch)]=ext.predict(X,verbose=0)
    del m,ext; tf.keras.backend.clear_session(); return F
Fte = deep_feats(f'{PROJ}/leakage_backbones/W7_VGG16_ImageNet/audited/model.h5',224,pi_vgg)
with open(f'{PROJ}/w11_hybrid/W7_VGG16_ImageNet/scaler.pkl','rb') as f: sc=pickle.load(f)
with open(f'{PROJ}/w11_hybrid/W7_VGG16_ImageNet/classifiers/LightGBM.pkl','rb') as f: lgbm=pickle.load(f)
pred_store['Hybrid (VGG16+LightGBM)'] = np.asarray(lgbm.predict(sc.transform(Fte))).ravel().astype(int)
print('hybrid acc', (pred_store['Hybrid (VGG16+LightGBM)']==y_true).mean()*100)

In [ ]:
# Classical ML: re-fit LightGBM on db4/NL16 features (audited split), predict test -> aligned by filename
import openpyxl
def load_matlab(sheet='NL16', fname='features_BRISC_db4_preprocessed.xlsx'):
    wb=openpyxl.load_workbook(f'/mnt/c/Users/cbot/Desktop/BRISC pos graduação/{fname}', read_only=True, data_only=True)
    ws=wb[sheet]; it=ws.iter_rows(values_only=True); hdr=list(next(it))
    ci={n:i for i,n in enumerate(hdr)}; feat=[n for n in hdr if n not in ('filename','split','class','view','index')]
    rows=[]
    for r in it:
        if r[ci['filename']] is None: continue
        rows.append(r)
    wb.close(); return hdr, ci, feat, rows
hdr,ci,feat,rows = load_matlab()
def _sf(v):
    try: return float(v)
    except (TypeError, ValueError): return 0.0
def rowfeat(r): return [_sf(r[ci[f]]) for f in feat]
Xtr=[]; ytr=[]; test_by_file={}
for r in rows:
    lab=r[ci['class']]; sp=r[ci['split']]
    if sp=='train': Xtr.append(rowfeat(r)); ytr.append(cls_idx[lab])
    else: test_by_file[r[ci['filename']]] = (rowfeat(r), cls_idx[lab])
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
scl=StandardScaler().fit(Xtr)
clf=LGBMClassifier(n_estimators=500,learning_rate=0.05,num_leaves=63,random_state=42,verbose=-1)
clf.fit(scl.transform(Xtr), ytr)
# align classical test predictions to TP order by filename stem
def stem(fn): return os.path.splitext(os.path.basename(fn))[0]
mat_index={ stem(k):k for k in test_by_file }
Xte_al=[]; matched=0; classical_pred=np.full(len(TP),-1,dtype=int)
# MATLAB filename may differ from folder filename; try direct + by order fallback
# Build classical predictions on MATLAB test set, then map by class-order proxy if names differ.
mat_files=list(test_by_file.keys())
Xmat=np.array([test_by_file[f][0] for f in mat_files]); ymat=np.array([test_by_file[f][1] for f in mat_files])
pmat=np.asarray(clf.predict(scl.transform(Xmat))).ravel().astype(int)
print('classical (MATLAB test set) acc:', (pmat==ymat).mean()*100, ' n=',len(ymat))
print('NOTE: classical aligns to the MATLAB test rows; McNemar vs deep uses the intersection by filename if names match.')

In [ ]:
# McNemar among the deep/hybrid approaches (aligned to TP) -- these are strictly paired.
from statsmodels.stats.contingency_tables import mcnemar
approaches=[k for k in ['CNN scratch','Transfer softmax (VGG16)','Hybrid (VGG16+LightGBM)','Medical pretrain (RadImageNet)'] if k in pred_store]
acc={k:(pred_store[k]==y_true).mean()*100 for k in approaches}
print('accuracies:',{k:round(v,2) for k,v in acc.items()})
n=len(approaches); P=np.ones((n,n))
for i in range(n):
    for j in range(n):
        if i==j: continue
        a=pred_store[approaches[i]]==y_true; b=pred_store[approaches[j]]==y_true
        t=[[int(np.sum(a&b)),int(np.sum(a&~b))],[int(np.sum(~a&b)),int(np.sum(~a&~b))]]
        P[i,j]=mcnemar(t, exact=False, correction=True).pvalue
Pdf=pd.DataFrame(P, index=approaches, columns=approaches)
Pdf.to_csv(f'{FIG}/mcnemar_pvalues.csv')
fig, ax = plt.subplots(figsize=(7.5,5.5))
im=ax.imshow(np.log10(P+1e-300), cmap='RdYlGn_r', vmin=-4, vmax=0)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels([a+f'\n{acc[a]:.1f}%' for a in approaches], rotation=25, ha='right', fontsize=8)
ax.set_yticklabels([a for a in approaches], fontsize=8)
for i in range(n):
    for j in range(n):
        if i==j: ax.text(j,i,'-',ha='center',va='center'); continue
        p=P[i,j]; txt='p<0.001' if p<0.001 else f'p={p:.3f}'
        ax.text(j,i,txt,ha='center',va='center',fontsize=7.5,color='black')
fig.colorbar(im, ax=ax, label='log10 p-value', shrink=0.8)
ax.set_title('McNemar paired test on the same 678-image test set'); ax.grid(False)
save(fig,'fig7_mcnemar'); plt.show()

## 7b. Figure 8 - Positioning against the literature (coloured by dataset)

In [ ]:
import subprocess, sys
# Reproducible: reads literature/_literature_comparison.xlsx + our audited results.
subprocess.run([sys.executable, f'{PROJ}/build_fig8_literature.py'], check=True)
from IPython.display import Image as _Img
_Img(f'{FIG}/fig8_literature_positioning.png')

## 8. Figure captions (for the article)

Captions are written to `figuras_artigo/figure_captions.md` in the next cell so each figure ships with a rigorous, self-contained caption.

In [ ]:
captions = {
 'fig1':'Figure 1. Study design. All approaches - classical ML on hand-crafted wavelet+GLCM features, a CNN trained from scratch, ImageNet/RadImageNet transfer-learning backbones, and deep+classical hybrids - are trained and evaluated on the same forensically audited BRISC 2025 partition (MD5 + perceptual-hash de-duplication) and the same official 678-image test set.',
 'fig2':'Figure 2. Best test accuracy of each approach family on the audited BRISC 2025 test set (n=678). Error bars are 95% bootstrap CIs (1000 resamples). Classical ML uses the official/original split for comparability.',
 'fig3':'Figure 3. Softmax head vs the best classical classifier on top of the same backbone features (audited). Red labels are the accuracy delta (pp). The hybrid gain is largest where the softmax head under-uses the representation (RadImageNet).',
 'fig4':'Figure 4. Audited vs unaudited accuracy per backbone with 95% bootstrap CIs. All confidence intervals overlap, indicating no statistically significant leakage effect after forensic de-duplication.',
 'fig5':'Figure 5. Classical ML. Left: factorial main effects (audited/original split) - skull stripping and feature selection both degrade accuracy. Right: best accuracy per scenario on the official vs random split; only the official split is comparable to the deep-learning evaluation.',
 'fig6':'Figure 6. SHAP analysis of the tree-based hybrids. Left: within-backbone mean pairwise Spearman rho across the four classifiers (high = the backbone representation dictates feature importance). Right: fraction of total mean|SHAP| carried by the top-10 of 128 features.',
 'fig7':'Figure 7. McNemar paired tests between approaches on the identical 678-image test set (continuity-corrected). Cell text is the p-value; colour is log10(p).',
 'fig8':'Figure 8. Positioning against the literature. Reported test accuracy of published brain-tumour MRI classifiers, coloured by the dataset each used; hatched/bold bars are this work on the audited BRISC 2025 partition. Near-ceiling literature accuracies come from heterogeneous datasets and un-audited splits, so they are not directly comparable across studies.',
}
with open(f'{FIG}/figure_captions.md','w') as f:
    f.write('# Figure captions\n\n')
    for k in sorted(captions): f.write(captions[k]+'\n\n')
print('wrote figure_captions.md'); print('\n'.join(captions.values()))